# Affordability Rebates Analysis

This notebook analyzes the impact of an "Affordability Rebates" reform on the enhanced_cps_2024 dataset.

The reform provides $3,000 per adult and $3,000 per dependent through the Recovery Rebate Credit mechanism for 2026.

In [1]:
from policyengine_us import Microsimulation
from policyengine_core.reforms import Reform
import pandas as pd
import numpy as np

ENHANCED_CPS = "hf://policyengine/policyengine-us-data/enhanced_cps_2024.h5"

## 1. Define Reform

In [2]:
reform = Reform.from_dict({
    "gov.irs.credits.recovery_rebate_credit.arpa.max.adult": {
        "2026-01-01.2026-12-31": 3000,
        "2027-01-01.2035-12-31": 0
    },
    "gov.irs.credits.recovery_rebate_credit.arpa.max.dependent": {
        "2026-01-01.2026-12-31": 3000,
        "2027-01-01.2100-12-31": 0
    },
    "gov.irs.credits.recovery_rebate_credit.arpa.phase_out.threshold.JOINT": {
        "2026-01-01.2026-12-31": 150000,
        "2027-01-01.2035-12-31": 0
    },
    "gov.irs.credits.recovery_rebate_credit.arpa.phase_out.threshold.SINGLE": {
        "2026-01-01.2026-12-31": 75000,
        "2027-01-01.2035-12-31": 0
    },
    "gov.irs.credits.recovery_rebate_credit.arpa.phase_out.threshold.SEPARATE": {
        "2026-01-01.2026-12-31": 75000,
        "2027-01-01.2035-12-31": 0
    },
    "gov.irs.credits.recovery_rebate_credit.arpa.phase_out.threshold.SURVIVING_SPOUSE": {
        "2026-01-01.2026-12-31": 75000,
        "2027-01-01.2035-12-31": 0
    },
    "gov.irs.credits.recovery_rebate_credit.arpa.phase_out.threshold.HEAD_OF_HOUSEHOLD": {
        "2026-01-01.2026-12-31": 112500,
        "2027-01-01.2035-12-31": 0
    }
}, country_id="us")

print("Reform defined: Affordability Rebates")
print("  - $3,000 per adult")
print("  - $3,000 per dependent")
print("  - Phase-out thresholds: $75k (Single), $112.5k (HoH), $150k (Joint)")

Reform defined: Affordability Rebates
  - $3,000 per adult
  - $3,000 per dependent
  - Phase-out thresholds: $75k (Single), $112.5k (HoH), $150k (Joint)


## 2. Reform Impact Analysis

In [3]:
# Create baseline and reformed simulations
baseline = Microsimulation(dataset=ENHANCED_CPS)
reformed = Microsimulation(reform=reform, dataset=ENHANCED_CPS)

In [4]:
# Calculate household net income
baseline_income = baseline.calc("household_net_income", period=2026, map_to="household")
reformed_income = reformed.calc("household_net_income", period=2026, map_to="household")
difference_income = reformed_income - baseline_income

print("Household Net Income Impact:")
print(f"  Baseline mean: ${baseline_income.mean():,.0f}")
print(f"  Reformed mean: ${reformed_income.mean():,.0f}")
print(f"  Average gain per household: ${difference_income.mean():,.0f}")

Household Net Income Impact:
  Baseline mean: $116,539
  Reformed mean: $121,742
  Average gain per household: $5,203


In [5]:
# Total cost of the program
total_cost = difference_income.sum()
print(f"\nTotal Program Cost:")
print(f"  ${total_cost:,.0f}")
print(f"  ${total_cost / 1e9:.2f} billion")


Total Program Cost:
  $750,756,963,002
  $750.76 billion


In [6]:
# Number of households receiving benefits
beneficiaries = (difference_income > 0)
beneficiary_count = beneficiaries.sum()
beneficiary_share = beneficiaries.mean()

print(f"\nBeneficiary Analysis:")
print(f"  Households receiving benefits: {beneficiary_count:,.0f}")
print(f"  Share of households benefiting: {beneficiary_share:.1%}")


Beneficiary Analysis:
  Households receiving benefits: 118,223,733
  Share of households benefiting: 81.9%


In [7]:
# Average benefit among those receiving
difference_values = difference_income.values
weights_hh = baseline.calc("household_weight", period=2026).values

beneficiary_mask = difference_values > 0
avg_benefit_among_beneficiaries = np.average(
    difference_values[beneficiary_mask], 
    weights=weights_hh[beneficiary_mask]
)

print(f"\nAverage benefit among beneficiaries: ${avg_benefit_among_beneficiaries:,.0f}")


Average benefit among beneficiaries: $6,350


In [8]:
# Distribution of benefits by income bracket
agi_baseline = baseline.calc("adjusted_gross_income", period=2026, map_to="household").values

income_brackets = [
    (0, 10000, "$0-$10k"),
    (10000, 25000, "$10k-$25k"),
    (25000, 50000, "$25k-$50k"),
    (50000, 75000, "$50k-$75k"),
    (75000, 100000, "$75k-$100k"),
    (100000, 150000, "$100k-$150k"),
    (150000, 200000, "$150k-$200k"),
    (200000, float('inf'), "$200k+")
]

print("\n" + "="*80)
print("BENEFIT DISTRIBUTION BY INCOME BRACKET")
print("="*80)

benefit_by_bracket = []
for lower, upper, label in income_brackets:
    mask = (agi_baseline >= lower) & (agi_baseline < upper)
    if mask.sum() > 0:
        bracket_benefit = np.sum(difference_values[mask] * weights_hh[mask])
        bracket_households = weights_hh[mask].sum()
        avg_benefit = bracket_benefit / bracket_households if bracket_households > 0 else 0
        benefiting_in_bracket = weights_hh[mask & beneficiary_mask].sum()
        pct_benefiting = benefiting_in_bracket / bracket_households * 100 if bracket_households > 0 else 0
        
        benefit_by_bracket.append({
            "Income Bracket": label,
            "Total Benefit": f"${bracket_benefit/1e9:.2f}B",
            "Avg Benefit": f"${avg_benefit:,.0f}",
            "% Benefiting": f"{pct_benefiting:.1f}%"
        })

benefit_df = pd.DataFrame(benefit_by_bracket)
print(benefit_df.to_string(index=False))
print("="*80)


BENEFIT DISTRIBUTION BY INCOME BRACKET
Income Bracket Total Benefit Avg Benefit % Benefiting
       $0-$10k      $141.82B      $5,042        98.3%
     $10k-$25k       $88.38B      $5,884        98.9%
     $25k-$50k      $129.80B      $5,362        94.1%
     $50k-$75k      $108.19B      $6,195        97.1%
    $75k-$100k       $96.14B      $6,404        80.8%
   $100k-$150k      $129.86B      $7,150        83.0%
   $150k-$200k       $22.53B      $2,231        29.4%
        $200k+       $23.30B      $1,570        30.2%


In [9]:
# Distribution by filing status
filing_status = baseline.calc("filing_status", period=2026, map_to="tax_unit").values
tax_unit_weight = baseline.calc("tax_unit_weight", period=2026).values

baseline_tax_unit_income = baseline.calc("tax_unit_net_income", period=2026).values
reformed_tax_unit_income = reformed.calc("tax_unit_net_income", period=2026).values
tax_unit_difference = reformed_tax_unit_income - baseline_tax_unit_income

filing_status_names = {
    1: "SINGLE",
    2: "JOINT",
    3: "SEPARATE",
    4: "HEAD_OF_HOUSEHOLD",
    5: "SURVIVING_SPOUSE"
}

print("\n" + "="*80)
print("BENEFIT DISTRIBUTION BY FILING STATUS")
print("="*80)

filing_data = []
for status_code, status_name in filing_status_names.items():
    mask = filing_status == status_code
    if mask.sum() > 0:
        status_benefit = np.sum(tax_unit_difference[mask] * tax_unit_weight[mask])
        status_count = tax_unit_weight[mask].sum()
        avg_benefit = status_benefit / status_count if status_count > 0 else 0
        benefiting = tax_unit_weight[mask & (tax_unit_difference > 0)].sum()
        pct_benefiting = benefiting / status_count * 100 if status_count > 0 else 0
        
        filing_data.append({
            "Filing Status": status_name,
            "Tax Units": f"{status_count:,.0f}",
            "Total Benefit": f"${status_benefit/1e9:.2f}B",
            "Avg Benefit": f"${avg_benefit:,.0f}",
            "% Benefiting": f"{pct_benefiting:.1f}%"
        })

filing_df = pd.DataFrame(filing_data)
print(filing_df.to_string(index=False))
print("="*80)

ValueError: Variable tax_unit_net_income does not exist.

In [ ]:
# Impact on households with/without children
is_child = baseline.calc("is_child", period=2026, map_to="person")
household_id = baseline.calc("household_id", period=2026, map_to="person")
household_weight_person = baseline.calc("household_weight", period=2026, map_to="person")

df_households = pd.DataFrame({
    'household_id': household_id.values,
    'is_child': is_child.values,
    'household_weight': household_weight_person.values
})

children_per_household = df_households.groupby('household_id').agg({
    'is_child': 'sum',
    'household_weight': 'first'
}).reset_index()

has_children = children_per_household.set_index('household_id')['is_child'] > 0
household_ids = baseline.calc("household_id", period=2026, map_to="household").values

# Map has_children to households
has_children_map = has_children.reindex(household_ids).fillna(False).values

# Households with children
with_children_mask = has_children_map
without_children_mask = ~has_children_map

benefit_with_children = np.sum(difference_values[with_children_mask] * weights_hh[with_children_mask])
benefit_without_children = np.sum(difference_values[without_children_mask] * weights_hh[without_children_mask])

count_with_children = weights_hh[with_children_mask].sum()
count_without_children = weights_hh[without_children_mask].sum()

avg_with_children = benefit_with_children / count_with_children if count_with_children > 0 else 0
avg_without_children = benefit_without_children / count_without_children if count_without_children > 0 else 0

print("\n" + "="*80)
print("IMPACT BY HOUSEHOLD COMPOSITION")
print("="*80)
print(f"Households WITH children:")
print(f"  Count: {count_with_children:,.0f}")
print(f"  Total benefit: ${benefit_with_children/1e9:.2f}B")
print(f"  Average benefit: ${avg_with_children:,.0f}")
print(f"\nHouseholds WITHOUT children:")
print(f"  Count: {count_without_children:,.0f}")
print(f"  Total benefit: ${benefit_without_children/1e9:.2f}B")
print(f"  Average benefit: ${avg_without_children:,.0f}")
print("="*80)

## 3. Summary

In [ ]:
# Create summary table
summary_data = {
    'Metric': [
        'Total Program Cost',
        'Households Benefiting',
        'Share of Households Benefiting',
        'Average Benefit (all households)',
        'Average Benefit (beneficiaries only)',
        'Benefit to HH with Children',
        'Benefit to HH without Children',
        'Avg Benefit - HH with Children',
        'Avg Benefit - HH without Children'
    ],
    'Value': [
        f"${total_cost/1e9:.2f}B",
        f"{beneficiary_count:,.0f}",
        f"{beneficiary_share:.1%}",
        f"${difference_income.mean():,.0f}",
        f"${avg_benefit_among_beneficiaries:,.0f}",
        f"${benefit_with_children/1e9:.2f}B",
        f"${benefit_without_children/1e9:.2f}B",
        f"${avg_with_children:,.0f}",
        f"${avg_without_children:,.0f}"
    ]
}

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*70)
print("AFFORDABILITY REBATES - SUMMARY")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)

# Save summary
summary_df.to_csv('affordability_rebates_summary.csv', index=False)
print("\nSummary saved to: affordability_rebates_summary.csv")

In [ ]:
# Save benefit by income bracket
benefit_df.to_csv('affordability_rebates_by_income.csv', index=False)
print("Income bracket analysis saved to: affordability_rebates_by_income.csv")

# Save benefit by filing status
filing_df.to_csv('affordability_rebates_by_filing_status.csv', index=False)
print("Filing status analysis saved to: affordability_rebates_by_filing_status.csv")